# Summary (hours + lines of code across x files)

*Please note: Not everyone were logging their hours, it was mainly used for a potential enforcer in showing who contributed and not, we were mostly sitting together when working on this project.. and so everyone in this group spent atleast 70 hours on this project.

In [6]:
import pandas as pd
from datetime import datetime
import os

hours_log = pd.read_csv("hours.csv")
hours_log["Date"] = pd.to_datetime(hours_log["Date"], format="%d.%m.%Y")
hours_log["Start"] = hours_log["Start"].str.replace(".", ":")
hours_log["End"] = hours_log["End"].str.replace(".", ":")
hours_log["Start"] = pd.to_datetime(hours_log["Start"], format="%H:%M")
hours_log["End"] = pd.to_datetime(hours_log["End"], format="%H:%M")
hours_log["Hours"] = (hours_log["End"] - hours_log["Start"]).dt.total_seconds() / 3600

# Clean up tags formatting 
hours_log["Tag"] = hours_log["Tag"].str.strip()

# Summarize hours by contributor
summary = hours_log.groupby("Name")["Hours"].sum().reset_index()
summary = summary.sort_values(by="Hours", ascending=False)

# Summarize hours by tag
tag_summary = hours_log.groupby("Tag")["Hours"].sum().reset_index()
tag_summary = tag_summary.sort_values(by="Hours", ascending=False)

# Projected final hours calculation (updated dates to 2026 Spring semester)
project_start = datetime(2026, 3, 25)
project_end = datetime(2026, 4, 20)
last_log_day = hours_log["Date"].max()

days_passed = (last_log_day - project_start).days + 1  # include start day
total_days = (project_end - project_start).days + 1

# Estimate final hours per author
summary["Hours per day"] = summary["Hours"] / days_passed
summary["Estimated final hours"] = summary["Hours per day"] * total_days

print(f"{'=' * 10} Development Hours Report {'=' * 10}")
display("# Total Hours by Contributor:", summary)
display("# Total Hours by Tag:", tag_summary)

========== Development Hours Report ==========


'# Total Hours by Contributor:'

,Name,Hours,Hours per day,Estimated final hours
3,vaakir,95.083333,2.641204,71.3125
0,BaalCat,56.250000,1.562500,42.1875
2,togun,21.716667,0.603241,16.2875
1,GabIsr,11.666667,0.324074,8.7500


'# Total Hours by Tag:'

,Tag,Hours
9,System 2,42.500000
8,System 1,37.583333
6,Refactor,35.500000
3,Presentation,24.250000
2,Planning,14.550000
1,Implementation,10.833333
5,Reading Theory & Coding,7.750000
7,Reproducibility,6.000000
0,Fix,3.000000
4,Reading & Exploration,2.750000


In [8]:
from pathlib import Path

EXCLUDE_DIRS = {"node_modules", "__pycache__", "data", ".pytest_cache", ".conda", ".venv",".vscode"}
EXCLUDE_FILES = {
    "_momemtum.py",
    "_others.py",
    "_trend.py",
    "_volatility.py",
    "_volume.py",
}
CODE_EXTENSIONS = {".py", ".ts", ".tsx", ".js", ".jsx", ".html", ".css"}


def count_lines(root: Path):
    total = 0
    file_count = 0

    for path in root.iterdir():
        if path.is_dir():
            if path.name in EXCLUDE_DIRS:
                continue
            sub_total, sub_files = count_lines(path)
            total += sub_total
            file_count += sub_files

        else:
            if path.name in EXCLUDE_FILES:
                continue

            if path.suffix in CODE_EXTENSIONS:
                try:
                    with path.open("r", encoding="utf-8", errors="ignore") as f:
                        lines = sum(1 for _ in f)
                        total += lines
                        # print(lines, path) # debug where what kinda illigal file was included
                        file_count += 1
                except OSError:
                    pass

    return total, file_count


if __name__ == "__main__":
    root = Path("../").resolve()
    total_lines, total_files = count_lines(root)
    print(f"Total lines: {total_lines}, Total files: {total_files}")

Total lines: 9352, Total files: 56
